In [1]:
import pandas as pd
# https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_OHXDEN.XPT
df = pd.read_sas("../data/raw/P_OHXDEN.XPT")
ctc_cols = [c for c in df.columns if c.endswith('CTC')]
has_untreated_coronal = (
    df[ctc_cols]
    .apply(lambda row: (row == b'U').any(), axis=1)
)
has_root_caries = df["OHXRCAR"] == 1
df["has_caries"] = (has_root_caries | has_untreated_coronal).astype(int)
tc_cols = [c for c in df.columns if c.endswith("TC") and not c.endswith("CTC")]
df["n_missing_teeth"] = (df[tc_cols] == 4).sum(axis=1)
df["n_filled_teeth"] = (df[ctc_cols] == b'F').sum(axis=1)
df["n_untreated_teeth"] = (df[ctc_cols] == b'U').sum(axis=1)
df["has_root_caries"] = (df["OHXRCAR"] == 1).astype(int)
features = [
    "SEQN",
    "has_caries",
    "n_missing_teeth",
    "n_filled_teeth",
    "n_untreated_teeth",
    "has_root_caries",
]

df = df[features]

# https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.XPT
demo_df = pd.read_sas("../data/raw/P_DEMO.XPT")
demo_df = demo_df[["SEQN", "RIDAGEYR", "RIAGENDR"]]
demo_df['is_female'] = (demo_df['RIAGENDR']==2).astype(int)
demo_df = demo_df.drop('RIAGENDR', axis=1)
df = df.merge(demo_df, on='SEQN', how='inner')

df.head()

,SEQN,has_caries,n_missing_teeth,n_filled_teeth,n_untreated_teeth,has_root_caries,RIDAGEYR,is_female
0,109263.0,1,12,0,8,0,2.0,0
1,109264.0,0,4,0,0,0,13.0,1
2,109265.0,1,14,0,10,0,2.0,0
3,109266.0,0,0,8,0,0,29.0,1
4,109269.0,1,12,0,8,0,2.0,0


In [2]:
# https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SMQ.XPT
smoker_df = pd.read_sas("../data/raw/P_SMQ.XPT")
smoker_df = smoker_df[['SEQN', 'SMQ020']]
smoker_df['ever_smoked'] = (smoker_df['SMQ020']==1).astype(int)
df = df.merge(smoker_df, on='SEQN', how='inner')
df.head()

,SEQN,has_caries,n_missing_teeth,n_filled_teeth,n_untreated_teeth,has_root_caries,RIDAGEYR,is_female,SMQ020,ever_smoked
0,109264.0,0,4,0,0,0,13.0,1,NaN,0
1,109266.0,0,0,8,0,0,29.0,1,2.0,0
2,109271.0,1,18,0,0,1,49.0,0,1.0,1
3,109273.0,0,10,10,0,0,36.0,0,1.0,1
4,109274.0,0,32,0,0,0,68.0,0,2.0,0


In [12]:
X = df[
    [
        "RIDAGEYR",
        "is_female",
        "n_missing_teeth",
        "n_filled_teeth",
        "ever_smoked"
    ]
]

y = (df["has_root_caries"] == 1).astype(int)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3, random_state=101)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000,solver="lbfgs")
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

import numpy as np
from sklearn.metrics import recall_score, precision_score, confusion_matrix, classification_report

threshold = 0.10
y_pred_tuned = (y_prob >= threshold).astype(int)

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))


[[1753 1040]
 [ 100  230]]
              precision    recall  f1-score   support

           0       0.95      0.63      0.75      2793
           1       0.18      0.70      0.29       330

    accuracy                           0.63      3123
   macro avg       0.56      0.66      0.52      3123
weighted avg       0.87      0.63      0.71      3123

